# 📊 Visualization Cheatsheet — DSI SquarePoint

Полная шпаргалка по визуализации для Dataset Interview.  
Для каждого графика: **когда применять**, **что смотреть**, **параметры**.  
В конце — **готовый copy-paste шаблон EDA**.

---

## Содержание

1. [Setup](#1-setup)  
2. [Распределения — одна переменная](#2-distributions)  
3. [Категориальные переменные](#3-categorical)  
4. [Связь двух переменных](#4-bivariate)  
5. [Корреляции](#5-correlations)  
6. [Таргет vs признаки](#6-target-vs-features)  
7. [Временные ряды](#7-time-series)  
8. [Диагностика модели — Residuals](#8-residuals)  
9. [Feature Importance](#9-feature-importance)  
10. [🚀 Copy-paste шаблон EDA](#10-template)


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, mannwhitneyu, shapiro
from scipy import stats as scipy_stats

# ── Seaborn theme ──────────────────────────────────────────────────────────
# style:   'whitegrid' | 'darkgrid' | 'white' | 'ticks'
# palette: 'muted' | 'deep' | 'pastel' | 'colorblind' (доступнее для дальтоников)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# ── Matplotlib defaults ───────────────────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 5)   # размер по умолчанию
plt.rcParams['axes.spines.top']   = False  # убираем верхнюю рамку
plt.rcParams['axes.spines.right'] = False  # убираем правую рамку

pd.set_option('display.float_format', '{:.4f}'.format)
SEED = 42
print('Setup OK')

## 2. Распределения — одна переменная

Первое что смотришь на любом датасете. Цель — понять форму, скос, хвосты, выбросы.


### 2.1 `sns.histplot` — гистограмма с KDE

**Когда:** всегда, для любой числовой переменной. Основной инструмент.

**Что смотреть:** симметрия, хвосты, мультимодальность (2+ пика → возможно смесь популяций).

In [ ]:
# Создаём синтетические данные для демонстрации
np.random.seed(SEED)
demo = pd.DataFrame({
    'normal':      np.random.normal(50, 10, 1000),
    'right_skewed': np.random.exponential(10, 1000),
    'bimodal':     np.concatenate([np.random.normal(30, 5, 500),
                                    np.random.normal(70, 5, 500)]),
    'category':    np.random.choice(['A','B','C'], 1000)
})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# bins      — количество столбцов. 'auto' автоматически, 30-50 обычно хорошо
# kde=True  — накладывает кривую плотности (Kernel Density Estimate)
#             помогает видеть форму независимо от binning
# color     — цвет заливки
# edgecolor — цвет границ столбцов ('white' даёт чёткое разделение)
# stat      — 'count' | 'frequency' | 'density' | 'probability'

sns.histplot(demo['normal'],       bins=40, kde=True, color='steelblue',
             edgecolor='white', ax=axes[0])
axes[0].set_title(f'Normal  skew={demo["normal"].skew():.2f}')

sns.histplot(demo['right_skewed'], bins=40, kde=True, color='steelblue',
             edgecolor='white', ax=axes[1])
axes[1].set_title(f'Right-skewed  skew={demo["right_skewed"].skew():.2f}  → log-transform')

sns.histplot(demo['bimodal'],      bins=40, kde=True, color='steelblue',
             edgecolor='white', ax=axes[2])
axes[2].set_title(f'Bimodal  skew={demo["bimodal"].skew():.2f}  → проверь субпопуляции')

sns.despine()
plt.suptitle('sns.histplot — три типичных случая', y=1.02)
plt.tight_layout()
plt.show()

# Правило интерпретации skewness:
print('skew > 1  → сильный правый скос → log1p трансформация')
print('skew < -1 → сильный левый скос  → потолочный эффект (ceiling effect)')
print('|skew| < 1 → приемлемо для линейных моделей')

### 2.2 `sns.boxplot` — ящик с усами

**Когда:** выбросы, медиана, IQR. Хорошо для сравнения нескольких групп.

**Что смотреть:** длина усов (разброс), точки за усами (выбросы), позиция медианы внутри ящика (симметрия).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Одна переменная
# width   — ширина ящика (0.4 компактнее, 0.8 шире)
# flierprops — стиль точек-выбросов
# linewidth  — толщина линий
sns.boxplot(
    y=demo['right_skewed'],
    color='steelblue',
    width=0.4,
    flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5},
    ax=axes[0]
)
axes[0].set_title('Boxplot — одна переменная')
axes[0].set_ylabel('right_skewed')

# Группировка по категории — главное применение boxplot
# x       — категориальная ось
# y       — числовая ось
# palette — цвета для каждой группы
# order   — порядок категорий
sns.boxplot(
    data=demo,
    x='category', y='normal',
    palette='muted',
    order=['A', 'B', 'C'],
    width=0.5,
    ax=axes[1]
)
axes[1].set_title('Boxplot — сравнение групп\n→ видим различия медиан и разброса')

sns.despine()
plt.tight_layout()
plt.show()

### 2.3 `sns.violinplot` — скрипичный график

**Когда:** хочешь видеть полное распределение внутри каждой группы, не только квартили.

**Что смотреть:** форма «скрипки» — где плотность выше, там больше наблюдений. Толстая середина = много значений в этом диапазоне.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# inner — что рисовать внутри скрипки:
#   'box'      — мини-boxplot (по умолчанию)
#   'quartile' — три горизонтальные линии (Q1, median, Q3)
#   'point'    — все точки
#   None       — только контур
# split  — True = две половины скрипки для двух групп (нужен hue с 2 значениями)
# bw_adjust — сглаживание KDE (< 1 = детальнее, > 1 = сглаженнее)
sns.violinplot(
    data=demo,
    x='category', y='normal',
    palette='muted',
    inner='box',
    bw_adjust=0.8,
    ax=ax
)
ax.set_title('violinplot — распределение внутри каждой группы')
sns.despine()
plt.tight_layout()
plt.show()

## 3. Категориальные переменные

Частоты, пропорции, сравнение групп.


### 3.1 `sns.countplot` — частоты категорий

**Когда:** распределение категориальной переменной. Первый взгляд на баланс классов.

**Что смотреть:** дисбаланс (один класс сильно доминирует → нужна стратификация при сплите).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# order    — порядок категорий на оси (обычно сортируем по частоте)
# hue      — раскраска по дополнительной переменной
# stat     — 'count' (по умолчанию) | 'percent' | 'probability'
order = demo['category'].value_counts().index

# Вертикальный (x=категория)
sns.countplot(data=demo, x='category', order=order,
              color='steelblue', ax=axes[0])
axes[0].set_title('countplot вертикальный')
axes[0].bar_label(axes[0].containers[0])  # подписи значений на столбцах

# Горизонтальный (y=категория) — лучше когда много категорий или длинные названия
sns.countplot(data=demo, y='category', order=order,
              color='steelblue', ax=axes[1])
axes[1].set_title('countplot горизонтальный\n(лучше для длинных названий)')

sns.despine()
plt.tight_layout()
plt.show()

### 3.2 `sns.barplot` — среднее (или другая агрегация) по группам

**Когда:** сравниваешь среднее значение числовой переменной между категориями. Например: средний рейтинг по типу ресторана.

**Что смотреть:** планки ошибок (95% CI по умолчанию) — если CI перекрываются, разница статистически незначима.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

# estimator — функция агрегации: np.mean (default) | np.median | np.sum
# errorbar  — 'ci' (confidence interval) | 'sd' (std dev) | 'se' | None
# capsize   — размер шапок на планках ошибок
# order     — порядок категорий
sns.barplot(
    data=demo,
    x='category', y='normal',
    estimator=np.mean,
    errorbar='ci',       # 95% confidence interval
    capsize=0.1,
    palette='muted',
    order=['A', 'B', 'C'],
    ax=ax
)
ax.set_title('barplot — среднее по группам + 95% CI\n'
             'Перекрывающиеся CI → разница незначима')
ax.set_ylabel('mean(normal)')
sns.despine()
plt.tight_layout()
plt.show()

## 4. Связь двух переменных

Scatter plots, regression lines, pairplots.


### 4.1 `sns.scatterplot` — точечный график

**Когда:** связь двух числовых переменных.

**Что смотреть:** линейность, кластеры, выбросы, гетероскедастичность (расширяющийся «конус»).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# alpha  — прозрачность точек (0.2-0.4 при большом N, чтобы видеть плотность)
# s      — размер точек
# hue    — цвет по третьей переменной
# style  — форма маркера по категории
# size   — размер по числовой переменной
sns.scatterplot(
    data=demo, x='normal', y='right_skewed',
    alpha=0.3, s=15, color='steelblue',
    ax=axes[0]
)
axes[0].set_title('scatterplot — базовый')

# С раскраской по категории — видим кластеры
sns.scatterplot(
    data=demo, x='normal', y='bimodal',
    hue='category',
    alpha=0.4, s=15,
    palette='muted',
    ax=axes[1]
)
axes[1].set_title('scatterplot с hue — видим структуру по группам')

sns.despine()
plt.tight_layout()
plt.show()

### 4.2 `sns.regplot` — scatter + линия регрессии

**Когда:** хочешь одновременно показать точки и линейный тренд с доверительным интервалом.

**Что смотреть:** насколько точки ложатся на линию (R²); расширение CI на краях (мало данных там).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# scatter_kws — dict параметров для scatterplot внутри (alpha, s, color)
# line_kws    — dict параметров для линии (color, linewidth, linestyle)
# ci          — ширина доверительного интервала (95 по умолчанию, None = убрать)
# order       — степень полинома (1=линейная, 2=квадратичная)
# lowess=True — непараметрическое сглаживание вместо прямой
sns.regplot(
    data=demo, x='normal', y='right_skewed',
    scatter_kws={'alpha': 0.25, 's': 12, 'color': 'steelblue'},
    line_kws={'color': 'red', 'linewidth': 1.5, 'linestyle': '--'},
    ci=95,
    ax=axes[0]
)
axes[0].set_title('regplot — линейный тренд + 95% CI')

# lowess — когда связь нелинейна
sns.regplot(
    data=demo, x='normal', y='bimodal',
    scatter_kws={'alpha': 0.25, 's': 12, 'color': 'steelblue'},
    line_kws={'color': 'red', 'linewidth': 2},
    lowess=True,  # нелинейное сглаживание
    ax=axes[1]
)
axes[1].set_title('regplot — lowess (нелинейный тренд)\n→ используй когда кривая лучше прямой')

sns.despine()
plt.tight_layout()
plt.show()

### 4.3 `sns.pairplot` — матрица попарных графиков

**Когда:** быстрый обзор всех попарных зависимостей. Обычно на первых 6-8 признаках.

**Что смотреть:** линейные паттерны (→ Ridge будет работать), кластеры (→ возможна структура), диагональ (распределения).

In [ ]:
# Берём небольшой датасет для наглядности
small = demo[['normal', 'right_skewed', 'bimodal', 'category']].sample(300, random_state=SEED)

# hue     — раскраска по категории на всех графиках
# diag_kind — что на диагонали: 'hist' | 'kde'
# kind    — тип off-diagonal графиков: 'scatter' | 'reg' (с линией регрессии)
# plot_kws — параметры точек
# corner  — True = только нижний треугольник (быстрее)
g = sns.pairplot(
    small,
    hue='category',
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 15},
    corner=True          # только нижний треугольник — экономит место
)
g.figure.suptitle('pairplot — быстрый обзор всех связей', y=1.02)
plt.show()

## 5. Корреляции

Heatmap — главный инструмент для матрицы корреляций.


### 5.1 `sns.heatmap` — тепловая карта корреляций

**Когда:** всегда при EDA числовых признаков. Обязательный элемент DSI.

**Что смотреть:** тёмно-синие клетки (ρ ≈ 1) = мультиколлинеарность → Ridge обработает, но осторожно с интерпретацией коэффициентов. Красные (ρ ≈ -1) = сильная обратная связь.

In [ ]:
# Строим матрицу корреляций
num_demo = demo[['normal', 'right_skewed', 'bimodal']]
corr = num_demo.corr(method='spearman')  # spearman устойчивее к выбросам

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Полная матрица
# annot    — True = показать числа в ячейках
# fmt      — формат чисел ('.2f' = два знака после запятой)
# cmap     — цветовая карта:
#   'RdBu_r'   — красный (отрицательный) → белый → синий (положительный)
#   'coolwarm' — похожая, менее резкая
#   'YlOrRd'   — для только положительных значений (importance и т.п.)
# center   — значение в центре цветовой шкалы (0 для корреляций)
# vmin/vmax — границы шкалы (-1/1 для корреляций)
# square   — True = квадратные ячейки
# linewidths — толщина разделителей между ячейками
# annot_kws — параметры шрифта аннотаций
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    annot_kws={'size': 10},
    ax=axes[0]
)
axes[0].set_title('Heatmap — полная матрица')

# Только нижний треугольник — убираем дублирование
mask = np.triu(np.ones_like(corr, dtype=bool))  # True в верхнем треугольнике
sns.heatmap(
    corr, mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    annot_kws={'size': 10},
    ax=axes[1]
)
axes[1].set_title('Heatmap — нижний треугольник (стандарт для EDA)')

plt.tight_layout()
plt.show()

## 6. Таргет vs признаки

Самый важный раздел для DSI — показываешь что умеешь находить сигнал.


### 6.1 Корреляция признаков с таргетом — barplot

**Когда:** ранжирование признаков по силе связи с таргетом. Обязательный слайд.

**Что смотреть:** топ признаки, знак корреляции, расхождение Pearson/Spearman (→ нелинейность).

In [ ]:
# Создаём демо-таргет
demo['target'] = 0.6 * demo['normal'] + 0.3 * demo['bimodal'] + np.random.normal(0, 5, 1000)

feat_cols = ['normal', 'right_skewed', 'bimodal']
corr_df = pd.DataFrame({
    'pearson':  [demo[c].corr(demo['target'], method='pearson')  for c in feat_cols],
    'spearman': [demo[c].corr(demo['target'], method='spearman') for c in feat_cols],
}, index=feat_cols)
corr_df['divergence'] = corr_df['spearman'].abs() - corr_df['pearson'].abs()
corr_df = corr_df.sort_values('spearman', key=abs, ascending=False)
print(corr_df)

plot_df = corr_df.reset_index().rename(columns={'index': 'feature'})
plot_df['sign'] = plot_df['spearman'].apply(lambda x: 'positive' if x >= 0 else 'negative')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Spearman barplot
sns.barplot(
    data=plot_df.sort_values('spearman'),
    x='spearman', y='feature',
    hue='sign',
    palette={'positive': 'steelblue', 'negative': 'coral'},
    legend=False,
    ax=axes[0]
)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Spearman ρ с таргетом')
axes[0].set_xlabel('Spearman ρ')
axes[0].set_ylabel('')

# Divergence — где нелинейность
sns.barplot(
    data=plot_df.sort_values('divergence', ascending=False),
    x='divergence', y='feature',
    color='coral', ax=axes[1]
)
axes[1].set_title('Divergence (|Spearman| − |Pearson|)\n> 0.05 → нелинейная связь → деревья')
axes[1].set_xlabel('Divergence')
axes[1].set_ylabel('')

sns.despine()
plt.tight_layout()
plt.show()

### 6.2 `sns.regplot` grid — топ признаки vs таргет

**Когда:** глубже смотришь на каждый топ-признак.

**Что смотреть:** линейность, выбросы, гетероскедастичность. p-value на title — статистическая значимость.

In [ ]:
top_features = ['normal', 'bimodal', 'right_skewed']
TARGET = 'target'

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(top_features):
    valid = demo[[col, TARGET]].dropna()
    sample = valid.sample(min(500, len(valid)), random_state=SEED)  # сэмпл для скорости

    sns.regplot(
        data=sample, x=col, y=TARGET,
        scatter_kws={'alpha': 0.3, 's': 12, 'color': 'steelblue'},
        line_kws={'color': 'red', 'linewidth': 1.5, 'linestyle': '--'},
        ci=95,
        ax=axes[i]
    )
    rho, p = spearmanr(valid[col], valid[TARGET])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    axes[i].set_title(f'{col}\nρ={rho:.3f}  {sig}', fontsize=10)
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].set_ylabel(TARGET, fontsize=9)
    sns.despine(ax=axes[i])

plt.suptitle('Top Features vs Target  (* p<0.05, ** p<0.01, *** p<0.001)', y=1.02)
plt.tight_layout()
plt.show()

## 7. Временные ряды

Для датасетов с временной осью.


### 7.1 `sns.lineplot` — временной ряд

**Когда:** данные упорядочены по времени.

**Что смотреть:** тренд, сезонность, разрывы (пропуски), аномальные периоды.

In [ ]:
# Создаём демо-временной ряд
dates = pd.date_range('2020-01-01', periods=200, freq='D')
ts = pd.DataFrame({
    'date':  dates,
    'value': np.cumsum(np.random.randn(200)) + 100,
    'group': np.where(np.arange(200) < 100, 'pre', 'post')
})

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Простой line plot
# markers — True = маркеры на каждой точке (плохо при большом N)
# dashes  — {'pre': (4,2), 'post': (1,0)} — тип линии для каждой группы
# errorbar — 'sd' | 'ci' | None — полоса неопределённости
sns.lineplot(
    data=ts, x='date', y='value',
    color='steelblue', linewidth=1.5,
    ax=axes[0]
)
axes[0].set_title('lineplot — простой временной ряд')
axes[0].tick_params(axis='x', rotation=30)

# С группировкой — сравниваем периоды
sns.lineplot(
    data=ts, x='date', y='value',
    hue='group',
    palette={'pre': 'steelblue', 'post': 'coral'},
    linewidth=1.5,
    ax=axes[1]
)
axes[1].set_title('lineplot — сравнение периодов (pre/post)')
axes[1].tick_params(axis='x', rotation=30)

sns.despine()
plt.tight_layout()
plt.show()

## 8. Диагностика модели — Residuals

После обучения модели — обязательный раздел на DSI.


### 8.1 Residuals vs Predicted + Distribution + QQ

**Когда:** после любой регрессионной модели.

**Что смотреть:**
- **Residuals vs Predicted:** конус → гетероскедастичность; кривая → нелинейность  
- **Histogram:** смещение от нуля → систематическая ошибка  
- **QQ plot:** отклонения от диагонали → нарушение нормальности

In [ ]:
# Симулируем предсказания и ошибки
np.random.seed(SEED)
y_true = np.random.normal(50, 10, 300)
y_pred = y_true + np.random.normal(0, 3, 300)
residuals = y_true - y_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Residuals vs Predicted
# Норма: точки случайно разбросаны вокруг y=0 без паттерна
sns.scatterplot(
    x=y_pred, y=residuals,
    alpha=0.4, s=15, color='steelblue', ax=axes[0]
)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted\n'
                  'Конус → гетероскедастичность\n'
                  'Кривая → нелинейность')

# 2. Residual distribution
sns.histplot(residuals, bins=40, kde=True, color='steelblue',
             edgecolor='white', ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--')
axes[1].axvline(residuals.mean(), color='orange', linestyle='--',
                label=f'mean={residuals.mean():.3f}')
axes[1].set_title(f'Residual Distribution\nmean={residuals.mean():.3f}  std={residuals.std():.3f}')
axes[1].legend(fontsize=8)

# 3. QQ plot — нормальность остатков
# Точки на диагонали → нормальное распределение
# Загибы на концах → тяжёлые хвосты
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(residuals, dist='norm')
axes[2].plot(osm, osr, 'o', alpha=0.4, markersize=4, color='steelblue')
axes[2].plot(osm, slope * np.array(osm) + intercept, 'r--', linewidth=1.5)
axes[2].set_title(f'Normal Q-Q Plot\nR={r:.3f}  (1.0 = идеальная нормальность)')
axes[2].set_xlabel('Theoretical quantiles')
axes[2].set_ylabel('Sample quantiles')

sns.despine()
plt.tight_layout()
plt.show()

# Shapiro-Wilk тест нормальности (надёжен до n≈5000)
n_sw = min(len(residuals), 500)
stat, p = shapiro(residuals[:n_sw])
print(f'Shapiro-Wilk: W={stat:.4f}  p={p:.4f}')
print('Норма: остатки нормальны.' if p > 0.05
      else 'Остатки не нормальны → рассмотри Huber loss или квантильную регрессию.')

## 9. Feature Importance

После обучения модели — что и насколько влияет на предсказание.


### 9.1 Feature Importance barplot

**Когда:** после LightGBM / XGBoost — всегда.

**Что смотреть:** топ признаки, нет ли среди них явного leakage (признак занимает 70%+ — подозрительно).

In [ ]:
# Симулируем importance values
np.random.seed(SEED)
features = [f'feature_{i}' for i in range(15)]
importances = np.random.exponential(10, 15)
importances = importances / importances.sum() * 100  # нормируем в %

fi_df = pd.DataFrame({
    'feature': features,
    'importance_pct': importances
}).sort_values('importance_pct', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 6))

# Горизонтальный barplot — лучше читаются названия признаков
sns.barplot(
    data=fi_df,
    x='importance_pct', y='feature',
    color='steelblue',
    orient='h',     # горизонтальный
    ax=ax
)
ax.set_xlabel('% от суммарного Gain')
ax.set_ylabel('')
ax.set_title('Feature Importance (Gain)\n'
             'Один признак > 50% → проверь на leakage')
sns.despine()
plt.tight_layout()
plt.show()

---

## 10. 🚀 Copy-Paste шаблон EDA для DSI SquarePoint

Скопируй ячейки ниже в свой ноутбук, замени `TARGET` и `DATA_PATH` — и EDA готов.


### Шаг 0 — Setup (всегда первая ячейка)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, mannwhitneyu, shapiro
from scipy import stats as scipy_stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
pd.set_option('display.float_format', '{:.4f}'.format)

SEED   = 42
TARGET = 'YOUR_TARGET_HERE'   # ← ЗАМЕНИТЬ
DATA_PATH = 'data.csv'        # ← ЗАМЕНИТЬ

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape[0]:,} × {df.shape[1]}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

### Шаг 1 — Первичный осмотр

In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T


### Шаг 2 — Качество данных

In [ ]:
# ── Пропуски ────────────────────────────────────────────────────────────
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'pct':   df.isnull().mean() * 100
}).query('count > 0').sort_values('pct', ascending=False)

if len(missing) > 0:
    missing['action'] = missing['pct'].apply(
        lambda p: 'impute' if p < 5 else 'impute + flag' if p < 30 else 'consider drop'
    )
    print(missing.to_string())

    plot_m = missing.reset_index().rename(columns={'index': 'column'})
    fig, ax = plt.subplots(figsize=(10, max(3, len(missing)*0.4)))
    sns.barplot(data=plot_m, x='pct', y='column', color='steelblue', ax=ax)
    ax.axvline(5,  color='orange', linestyle='--', linewidth=1.2, label='5%')
    ax.axvline(30, color='red',    linestyle='--', linewidth=1.2, label='30%')
    ax.set(xlabel='Missing (%)', ylabel='', title='Missingness by Column')
    ax.legend(); sns.despine(); plt.tight_layout(); plt.show()
else:
    print('No missing values ✓')

# ── Дубликаты ────────────────────────────────────────────────────────────
n_dups = df.duplicated().sum()
print(f'Duplicates: {n_dups} ({n_dups/len(df)*100:.2f}%)')
if n_dups > 0:
    df = df.drop_duplicates().reset_index(drop=True)

# ── Выбросы (3×IQR) ──────────────────────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.tolist()
outlier_report = []
for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1-3*IQR) | (df[col] > Q3+3*IQR)).sum()
    if n_out > 0:
        outlier_report.append({'column': col, 'n': n_out, 'pct': round(n_out/len(df)*100,2)})
if outlier_report:
    print('Outliers (3×IQR):'); print(pd.DataFrame(outlier_report).to_string(index=False))
else:
    print('No outliers at 3×IQR ✓')

### Шаг 3 — Распределение таргета

In [ ]:
print(f'Target: {TARGET}')
print(f'Unique values: {df[TARGET].nunique()}')
print(df[TARGET].describe())
print(f'Skew: {df[TARGET].skew():.3f}  Kurt: {df[TARGET].kurt():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df[TARGET], bins=50, kde=True, color='steelblue',
             edgecolor='white', ax=axes[0])
axes[0].axvline(df[TARGET].mean(),   color='red',    linestyle='--', label=f'Mean={df[TARGET].mean():.2f}')
axes[0].axvline(df[TARGET].median(), color='orange', linestyle='--', label=f'Median={df[TARGET].median():.2f}')
axes[0].set(title=f'{TARGET} — Distribution'); axes[0].legend(fontsize=8)

sns.boxplot(y=df[TARGET], color='steelblue', width=0.4, ax=axes[1])
axes[1].set(title=f'{TARGET} — Boxplot')

sns.despine(); plt.tight_layout(); plt.show()

### Шаг 4 — Распределения числовых признаков

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), bins=40, kde=True, color='steelblue',
                 edgecolor='white', alpha=0.8, ax=axes[i])
    skew = df[col].skew()
    axes[i].set_title(f'{col}\nskew={skew:.2f}{" ⚠" if abs(skew)>1 else ""}', fontsize=9)
    axes[i].set_xlabel(''); sns.despine(ax=axes[i])

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Numeric Distributions  (⚠ = consider log-transform)', y=1.01)
plt.tight_layout(); plt.show()

### Шаг 5 — Категориальные признаки

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {len(cat_cols)}')

for col in cat_cols:
    n_unique = df[col].nunique()
    print(f'\n{col}: {n_unique} unique  {"⚠ HIGH CARDINALITY" if n_unique > 20 else ""}')
    print(df[col].value_counts().head(10).to_string())
    if n_unique <= 20:
        order = df[col].value_counts().index
        fig, ax = plt.subplots(figsize=(8, 3))
        sns.countplot(data=df, y=col, order=order, color='steelblue', ax=ax)
        ax.set(title=f'{col} — Value Counts', xlabel='Count', ylabel='')
        sns.despine(); plt.tight_layout(); plt.show()

### Шаг 6 — Корреляционная матрица

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(max(8, len(num_cols)), max(6, len(num_cols)-1)))
sns.heatmap(
    corr, mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, annot_kws={'size': 8},
    ax=ax
)
ax.set_title('Spearman Correlation Matrix (lower triangle)')
plt.tight_layout(); plt.show()

### Шаг 7 — Корреляция признаков с таргетом

In [ ]:
feat_cols = [c for c in num_cols if c != TARGET]
target_corr = pd.DataFrame({
    'pearson' : [df[c].corr(df[TARGET], method='pearson')  for c in feat_cols],
    'spearman': [df[c].corr(df[TARGET], method='spearman') for c in feat_cols],
}, index=feat_cols)
target_corr['abs_spearman'] = target_corr['spearman'].abs()
target_corr['divergence']   = target_corr['spearman'].abs() - target_corr['pearson'].abs()
target_corr = target_corr.sort_values('abs_spearman', ascending=False)
print(target_corr[['pearson','spearman','divergence']].to_string())

plot_df = target_corr.reset_index().rename(columns={'index': 'feature'})
plot_df['sign'] = plot_df['spearman'].apply(lambda x: 'pos' if x >= 0 else 'neg')

fig, ax = plt.subplots(figsize=(10, max(5, len(feat_cols)*0.35)))
sns.barplot(
    data=plot_df.sort_values('spearman'),
    x='spearman', y='feature',
    hue='sign', palette={'pos': 'steelblue', 'neg': 'coral'},
    legend=False, ax=ax
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title=f'Spearman Correlation with {TARGET}', xlabel='Spearman ρ', ylabel='')
sns.despine(); plt.tight_layout(); plt.show()

### Шаг 8 — Scatter: топ признаки vs таргет

In [ ]:
top_features = target_corr.head(6).index.tolist()
n = len(top_features)
n_cols = min(3, n)
n_rows = (n + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = np.array(axes).flatten()

for i, col in enumerate(top_features):
    valid  = df[[col, TARGET]].dropna()
    sample = valid.sample(min(2000, len(valid)), random_state=SEED)
    sns.regplot(
        data=sample, x=col, y=TARGET,
        scatter_kws={'alpha': 0.25, 's': 10, 'color': 'steelblue'},
        line_kws={'color': 'red', 'linewidth': 1.5, 'linestyle': '--'},
        ci=95, ax=axes[i]
    )
    rho, p = spearmanr(valid[col], valid[TARGET])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    axes[i].set_title(f'{col}\nρ={rho:.3f}  {sig}', fontsize=9)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel(TARGET, fontsize=8)
    sns.despine(ax=axes[i])

for j in range(n, len(axes)): axes[j].set_visible(False)
plt.suptitle(f'Top Features vs {TARGET}  (* p<0.05  ** p<0.01  *** p<0.001)', y=1.02)
plt.tight_layout(); plt.show()

---

## Quick Reference — когда что использовать

| Задача | График | Функция |
|---|---|---|
| Одна числовая переменная | Гистограмма + KDE | `sns.histplot(kde=True)` |
| Выбросы, медиана | Ящик с усами | `sns.boxplot` |
| Распределение внутри групп | Скрипка | `sns.violinplot` |
| Частоты категорий | Столбчатый | `sns.countplot` |
| Среднее по группам | Барплот + CI | `sns.barplot(errorbar='ci')` |
| Связь двух числовых | Scatter | `sns.scatterplot` |
| Scatter + тренд | Scatter + регрессия | `sns.regplot` |
| Все попарные связи | Матрица графиков | `sns.pairplot` |
| Матрица корреляций | Тепловая карта | `sns.heatmap` |
| Ранжирование признаков | Горизонт. барплот | `sns.barplot(orient='h')` |
| Временной ряд | Линейный график | `sns.lineplot` |
| Остатки модели | Scatter + hist + QQ | `sns.scatterplot` + `sns.histplot` + `scipy.probplot` |

**Правило Spearman vs Pearson:**  
Всегда считай оба. Если `|Spearman| − |Pearson| > 0.05` — связь нелинейная, деревья предпочтительнее Ridge.

**Правило significance:**  
`***` p < 0.001 / `**` p < 0.01 / `*` p < 0.05 / `n.s.` — всегда указывай на scatter plots.
